### Imports/installs

In [100]:
import re
import numpy as np
import pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

### Remove usernames/PII

In [79]:
USERNAME_PATTERNS = [r"@\w+", r"@\[[^\]]+\]", r"@\(.*?\)", r"@[^ \t\n\r\f\v]+"]
def remove_usernames(t):
    if not isinstance(t, str):
        return ""
    for pat in USERNAME_PATTERNS:
        t = re.sub(pat, " ", t)
    return re.sub(r"\s+", " ", t).strip()

### Change labels: text, bully, category

In [90]:
def normalize_labels_for_csv(df):
    df = df.copy()
    # make header handling robust
    df.columns = [c.strip() for c in df.columns]
    # required columns
    req = {"Text","Annotation","oh_label"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}. Found: {list(df.columns)}")

    # clean text and remove @usernames
    df["text"] = df["Text"].astype(str).map(remove_usernames)

    # Annotation: 0 or 1
    df["bully"] = pd.to_numeric(df["oh_label"], errors="coerce").fillna(0).astype(int)
    df["bully"] = df["bully"].clip(0,1)

    cats = df["Annotation"].astype(str).str.lower().str.strip()
    cats = np.where(df["bully"]==1,
                    np.where(cats.str.contains("rac"), "racism",
                             np.where(cats.str.contains("sex"), "sexism", "none")),
                    "none")
    df["category"] = cats

    return df[["text","bully","category"]]

### Training: train/val/test split, build pipelines, tune threshold

In [81]:
def split_sets(df, seed=42):
    train_val, test = train_test_split(df, test_size=0.15, random_state=seed, stratify=df["bully"])
    train, val = train_test_split(train_val, test_size=0.15/(1-0.15), random_state=seed, stratify=train_val["bully"])
    return train, val, test

In [82]:
def build_bin_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1,2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True,
            strip_accents="unicode",
            lowercase=True
        )),
        ("clf", LogisticRegression(
            class_weight="balanced",
            C=0.8,
            max_iter=3000,
            solver="liblinear"
        ))
    ])

In [83]:
def build_cat_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1,2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True,
            strip_accents="unicode",
            lowercase=True
        )),
        ("clf", LogisticRegression(
            class_weight="balanced",
            C=0.8,
            max_iter=3000,
            solver="liblinear"
        ))
    ])

In [84]:
def tune_threshold(bin_model, X_val, y_val):
    proba_pos = bin_model.predict_proba(X_val)[:,1]
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.2, 0.8, 121):
        pred = (proba_pos >= t).astype(int)
        f1 = f1_score(y_val, pred, pos_label=1)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

### Model evaluation

In [85]:
def eval_report(model, X, y, title):
    p = model.predict(X)
    print(f"\n== {title} ==")
    print("Accuracy:", f"{accuracy_score(y,p):.4f}")
    print("F1 (macro):", f"{f1_score(y,p,average='macro'):.4f}")
    print(classification_report(y,p,digits=4))

### Model Prediction

In [98]:
def predict_pipeline(texts, threshold=None, backstop=True, cat_conf=0.70):
    if threshold is None:
        threshold = t_star
    probs = bin_model.predict_proba(texts)[:,1]
    bpred = (probs >= threshold).astype(int)
    cats = np.array(["none"]*len(texts), dtype=object)

    if cat_model is not None:
        idx = np.where(bpred==1)[0]
        if len(idx):
            cats[idx] = cat_model.predict([texts[i] for i in idx])

        if backstop:
            tf = cat_model.named_steps["tfidf"]
            clf = cat_model.named_steps["clf"]
            Xc = tf.transform(texts)
            cprobs = clf.predict_proba(Xc)
            chat = clf.classes_[np.argmax(cprobs, axis=1)]
            cmax  = np.max(cprobs, axis=1)
            for i in range(len(texts)):
                if bpred[i]==0 and cmax[i] >= cat_conf and chat[i] in ("racism","sexism"):
                    bpred[i] = 1
                    cats[i]  = chat[i]

    # cast to plain Python types
    out = []
    for txt, b, c, p in zip(texts, bpred, cats, probs):
        out.append((txt, int(b), str(c), float(round(p,3))))
    return out

### Run: clean relabeled data, train/evaluate model

In [99]:
# 1) load and process
df_raw = pd.read_csv("twitter_parsed_dataset.csv")
df = normalize_labels_for_csv(df_raw)

# quick sanity
display(df.head())
print(df["bully"].value_counts(dropna=False))
print(df["category"].value_counts(dropna=False))

# 2) split
train, val, test = split_sets(df, seed=42)

# 3) train binary model
bin_model = build_bin_pipeline()
bin_model.fit(train["text"], train["bully"])
t_star, f1_star = tune_threshold(bin_model, val["text"], val["bully"])
print("Chosen threshold:", t_star, "F1@t:", f1_star)

# 4) train category model only on bullying rows
train_b = train[train["bully"]==1].query("category in ['racism','sexism']")
val_b   = val[val["bully"]==1].query("category in ['racism','sexism']")
test_b  = test[test["bully"]==1].query("category in ['racism','sexism']")

if all(len(x)>0 for x in [train_b,val_b,test_b]) and train_b["category"].nunique()==2:
    cat_model = build_cat_pipeline()
    cat_model.fit(train_b["text"], train_b["category"])
    eval_report(cat_model, val_b["text"], val_b["category"], "Category Validation")
    eval_report(cat_model, test_b["text"], test_b["category"], "Category Test")
else:
    print("Not enough bullying rows with both racism and sexism to train a category model.")

# 5) end‑to‑end predict
sample = ["@user you are so dumb",
          "Have a nice day",
          "women don’t belong here",
          "that policy is terrible"]

result = predict_pipeline(sample)
result

,text,bully,category
0,I read them in context.No change in meaning. T...,0,none
1,Now you idiots claim that people who tried to ...,0,none
2,"RT Call me sexist, but when I go to an auto pl...",1,sexism
3,"Wrong, ISIS follows the example of Mohammed an...",1,racism
4,#mkr No No No No No No,0,none


bully
0    11504
1     5347
Name: count, dtype: int64
category
none      11504
sexism     3377
racism     1970
Name: count, dtype: int64
Chosen threshold: 0.54 F1@t: 0.735573874445149

== Category Validation ==
Accuracy: 0.9364
F1 (macro): 0.9323
              precision    recall  f1-score   support

      racism     0.8711    0.9652    0.9157       287
      sexism     0.9793    0.9204    0.9489       515

    accuracy                         0.9364       802
   macro avg     0.9252    0.9428    0.9323       802
weighted avg     0.9406    0.9364    0.9371       802


== Category Test ==
Accuracy: 0.9364
F1 (macro): 0.9323
              precision    recall  f1-score   support

      racism     0.8766    0.9585    0.9157       289
      sexism     0.9753    0.9240    0.9489       513

    accuracy                         0.9364       802
   macro avg     0.9259    0.9412    0.9323       802
weighted avg     0.9397    0.9364    0.9370       802



[('@user you are so dumb', 0, 'none', 0.499),
 ('Have a nice day', 0, 'none', 0.314),
 ('women don’t belong here', 1, 'sexism', 0.462),
 ('that policy is terrible', 0, 'none', 0.332)]